In [2]:
# 0

import os
import wrds
import numpy as np
import pandas as pd
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

db = wrds.Connection(verbose=True)

START_DATE = '1976-01-01'

FVOL_COLS = ['actq','rectq','invtq','ppentq','atq','lctq','dlcq',
             'apq','dlttq','ltq','seqq','xoprq','cogsq','xsgaq']

postgresql://:@wrds-pgdata.wharton.upenn.edu:9737/wrds
err=OperationalError('(psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: server closed the connection unexpectedly\n\tThis probably means the server terminated abnormally\n\tbefore or while processing the request.\n')
postgresql://seanjeon:@wrds-pgdata.wharton.upenn.edu:9737/wrds
err=OperationalError('(psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: server closed the connection unexpectedly\n\tThis probably means the server terminated abnormally\n\tbefore or while processing the request.\n')


OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# 1

COMP_FILE = 'comp_fundq_raw.parquet'

if os.path.exists(COMP_FILE):
    comp_raw = pd.read_parquet(COMP_FILE)
else:
    cols = (['gvkey','datadate','rdq','fyearq','fqtr','saleq']
            + FVOL_COLS)
    q = f"""
        SELECT {', '.join(cols)}
        FROM comp.fundq
        WHERE indfmt='INDL' AND datafmt='STD' AND consol='C' AND popsrc='D'
          AND datadate >= '{START_DATE}'
    """
    comp_raw = db.raw_sql(q, date_cols=['datadate','rdq'])
    comp_raw = comp_raw.drop_duplicates(subset=['gvkey','datadate'])
    comp_raw = comp_raw.sort_values(['gvkey','datadate']).reset_index(drop=True)
    comp_raw.to_parquet(COMP_FILE)

print('comp_raw:', comp_raw.shape)


In [ ]:
# 2

CRSP_FILE = 'crsp_msf_raw.parquet'
DLST_FILE = 'crsp_msedelist.parquet'

if os.path.exists(CRSP_FILE):
    crsp_raw = pd.read_parquet(CRSP_FILE)
else:
    q = f"""
        SELECT a.permno, a.date, a.ret, a.prc, a.shrout,
               b.shrcd, b.exchcd, b.siccd
        FROM crsp.msf AS a
        LEFT JOIN crsp.msenames AS b
          ON a.permno=b.permno AND b.namedt<=a.date AND a.date<=b.nameendt
        WHERE a.date >= '{START_DATE}'
    """
    crsp_raw = db.raw_sql(q, date_cols=['date'])
    crsp_raw.to_parquet(CRSP_FILE)

if os.path.exists(DLST_FILE):
    dlst = pd.read_parquet(DLST_FILE)
else:
    qd = """
        SELECT permno, dlstdt, dlret, dlstcd
        FROM crsp.msedelist
        WHERE dlstcd IS NOT NULL
    """
    dlst = db.raw_sql(qd, date_cols=['dlstdt'])
    dlst.to_parquet(DLST_FILE)

print('crsp_raw:', crsp_raw.shape, ' dlst:', dlst.shape)

In [ ]:
# 3

LINK_FILE = 'ccm_link.parquet'

if os.path.exists(LINK_FILE):
    link = pd.read_parquet(LINK_FILE)
else:
    q = """
        SELECT gvkey, lpermno AS permno, linktype, linkprim, linkdt, linkenddt
        FROM crsp.ccmxpf_linktable
        WHERE linktype IN ('LU','LC') AND linkprim IN ('P','C')
    """
    link = db.raw_sql(q, date_cols=['linkdt','linkenddt'])
    link['linkenddt'] = link['linkenddt'].fillna(pd.Timestamp('2030-12-31'))
    link.to_parquet(LINK_FILE)

print('link:', link.shape)

In [ ]:
# 4

crsp_u = crsp_raw.copy()
crsp_u = crsp_u[crsp_u['shrcd'].isin([10, 11])]
crsp_u = crsp_u[crsp_u['exchcd'].isin([1, 2, 3])]
crsp_u = crsp_u[~crsp_u['siccd'].between(6000, 6999)]
crsp_u['permno'] = crsp_u['permno'].astype('int64')
crsp_u['ym'] = pd.to_datetime(crsp_u['date']).dt.to_period('M')
univ_fq = crsp_u[['permno','ym']].drop_duplicates()

lk = link.copy()
lk['permno'] = lk['permno'].astype('int64')

def prep(df, fvol_cols=FVOL_COLS):
    df = df.drop_duplicates(subset=['gvkey','datadate'])
    df = df.sort_values(['gvkey','datadate']).reset_index(drop=True)
    df['n_valid'] = df[fvol_cols].notna().sum(axis=1)
    df = df[df['n_valid'] >= 10].reset_index(drop=True)
    df = df.dropna(subset=['fyearq','fqtr'])
    df['qidx'] = df['fyearq'].astype('int64')*4 + (df['fqtr'].astype('int64')-1)
    df = df.drop_duplicates(subset=['gvkey','qidx'], keep='last')
    df = df.sort_values(['gvkey','qidx']).reset_index(drop=True)
    return df

def build_grid(df, fvol_cols=FVOL_COLS):
    rng = df.groupby('gvkey')['qidx'].agg(['min','max'])
    counts = (rng['max']-rng['min']+1).astype('int64')
    gvk = np.repeat(rng.index.values, counts.values)
    off = np.concatenate([np.arange(n) for n in counts.values])
    qf = np.repeat(rng['min'].values, counts.values) + off
    grid = pd.DataFrame({'gvkey': gvk, 'qidx': qf})
    d = grid.merge(df, on=['gvkey','qidx'], how='left').sort_values(['gvkey','qidx']).reset_index(drop=True)
    g = d.groupby('gvkey', sort=False)
    for c in fvol_cols:
        d[f'd_{c}'] = d[c] - g[c].shift(4)
    g = d.groupby('gvkey', sort=False)
    ind = []
    for c in fvol_cols:
        d[f'std_{c}'] = g[f'd_{c}'].rolling(8, min_periods=4).std().reset_index(level=0, drop=True)
    d['avg_sales'] = g['saleq'].rolling(8, min_periods=4).mean().reset_index(level=0, drop=True)
    d['avg_sales_valid'] = d['avg_sales'].where(d['avg_sales'] > 0)
    for c in fvol_cols:
        d[f'fvol_{c}'] = d[f'std_{c}'] / d['avg_sales_valid']
        ind.append(f'fvol_{c}')
    return d[d[c].notna() | True], ind

def build_dist(df, fvol_cols=FVOL_COLS):
    d = df.copy()
    g = d.groupby('gvkey', sort=False)
    prevq = g['qidx'].shift(4)
    has4 = (d['qidx'] - prevq) == 4
    for c in fvol_cols:
        d[f'd_{c}'] = np.where(has4, d[c] - g[c].shift(4), np.nan)
    ind = []
    d = d.sort_values(['gvkey','qidx']).reset_index(drop=True)
    g = d.groupby('gvkey', sort=False)
    def roll_std(s):
        return s.rolling(8, min_periods=4).std()
    def roll_mean(s):
        return s.rolling(8, min_periods=4).mean()
    for c in fvol_cols:
        d[f'std_{c}'] = g[f'd_{c}'].apply(roll_std).reset_index(level=0, drop=True)
    d['saleq_pos'] = d['saleq'].where(d['saleq'] > 0)
    g = d.groupby('gvkey', sort=False)
    d['avg_sales'] = g['saleq_pos'].apply(roll_mean).reset_index(level=0, drop=True)
    d['avg_sales_valid'] = d['avg_sales'].where(d['avg_sales'] > 0)
    for c in fvol_cols:
        d[f'fvol_{c}'] = d[f'std_{c}'] / d['avg_sales_valid']
        ind.append(f'fvol_{c}')
    return d, ind

base = prep(comp_raw)

def finalize(d, ind):
    d = d.copy()
    d['ym'] = pd.to_datetime(d['datadate']).dt.to_period('M')
    d['calq'] = pd.to_datetime(d['datadate']).dt.to_period('Q')
    cl = d.merge(lk[['gvkey','permno','linkdt','linkenddt']], on='gvkey', how='inner')
    cl = cl[(cl['datadate'] >= cl['linkdt']) & (cl['datadate'] <= cl['linkenddt'])]
    cl = cl.merge(univ_fq, on=['permno','ym'], how='inner')
    keep = cl[['gvkey','qidx']].drop_duplicates()
    d = d.merge(keep.assign(u=1), on=['gvkey','qidx'], how='left')
    d.loc[d['u'] != 1, ind] = np.nan
    ranked = d.groupby('calq')[ind].rank(pct=True)
    ranked.columns = [f'rank_{x}' for x in FVOL_COLS]
    d = pd.concat([d, ranked], axis=1)
    rcols = [f'rank_{x}' for x in FVOL_COLS]
    d['n_valid_rank'] = d[rcols].notna().sum(axis=1)
    d['FVOL'] = d[rcols].mean(axis=1)
    d.loc[d['n_valid_rank'] < 10, 'FVOL'] = np.nan
    return d

dg, ind_g = build_grid(base)
dd, ind_d = build_dist(base)
comp_grid = finalize(dg, ind_g)
comp_dist = finalize(dd, ind_d)

for nm, c in [('GRID', comp_grid), ('DIST', comp_dist)]:
    cc = c[c['datadate'] <= '2024-12-31']
    f = cc[cc['FVOL'].notna()]
    print(nm, 'avg firms/calq:', round(f.groupby('calq').size().mean(), 0))

comp = comp_dist[comp_dist['datadate'] <= '2024-12-31']
fvol = comp[comp['FVOL'].notna()][['gvkey','datadate','rdq','FVOL']].copy()

In [ ]:
# 5

crsp = crsp_raw.copy()
crsp = crsp[crsp['shrcd'].isin([10, 11])]
crsp = crsp[crsp['exchcd'].isin([1, 2, 3])]
crsp = crsp[~crsp['siccd'].between(6000, 6999)]
crsp['permno'] = crsp['permno'].astype('int64')
crsp['ret'] = pd.to_numeric(crsp['ret'], errors='coerce')

crsp['ym'] = pd.to_datetime(crsp['date']).dt.to_period('M')

d = dlst.copy()
d['permno'] = d['permno'].astype('int64')
d['ym'] = pd.to_datetime(d['dlstdt']).dt.to_period('M')
d['dlret'] = pd.to_numeric(d['dlret'], errors='coerce')
d = d.sort_values(['permno','ym']).drop_duplicates(['permno','ym'], keep='last')

crsp = crsp.merge(d[['permno','ym','dlret','dlstcd']], on=['permno','ym'], how='left')

last_obs = crsp.groupby('permno')['ym'].transform('max')
need_row = d.merge(
    crsp[['permno','ym']].drop_duplicates().assign(in_panel=1),
    on=['permno','ym'], how='left'
)
need_row = need_row[need_row['in_panel'].isna()].copy()

info = crsp.sort_values(['permno','ym']).groupby('permno').tail(1)[
    ['permno','shrcd','exchcd','siccd','prc','shrout']
]
add = need_row.merge(info, on='permno', how='inner')
add = add[add['shrcd'].isin([10,11]) & add['exchcd'].isin([1,2,3]) & ~add['siccd'].between(6000,6999)]
add['date'] = add['dlstdt']
add['ret'] = np.nan
add = add[['permno','date','ret','prc','shrout','shrcd','exchcd','siccd','ym','dlret','dlstcd']]

crsp = pd.concat([crsp, add], ignore_index=True)

has_dl = crsp['dlstcd'].notna()
perf = crsp['dlstcd'].between(500, 599) | crsp['dlstcd'].isin([574, 580, 584])
dlret = crsp['dlret'].copy()
dlret = dlret.mask(has_dl & dlret.isna() & perf, -0.30)
dlret = dlret.mask(has_dl & dlret.isna() & ~perf, 0.0)

crsp['ret_adj'] = crsp['ret']
crsp.loc[has_dl, 'ret_adj'] = (
    (1 + crsp.loc[has_dl, 'ret'].fillna(0)) * (1 + dlret.loc[has_dl]) - 1
)

crsp['mktcap'] = crsp['prc'].abs() * crsp['shrout']
crsp = crsp.sort_values(['permno','date']).reset_index(drop=True)
crsp = crsp[crsp['date'] <= '2024-12-31']

print('crsp:', crsp.shape)
print('delisting rows applied:', has_dl.sum())
print('appended delist-only rows:', len(add))
print('ret_adj != ret (where dl):', (crsp['ret_adj'] != crsp['ret']).sum())

In [ ]:
# 6

fv = fvol.merge(link, on='gvkey', how='inner')
fv = fv[(fv['datadate'] >= fv['linkdt']) & (fv['datadate'] <= fv['linkenddt'])]
fv = fv.dropna(subset=['rdq']).copy()
fv['permno'] = fv['permno'].astype('int64')

fv['avail'] = pd.to_datetime(fv['rdq']).dt.to_period('Q').dt.end_time.dt.normalize()
fv = fv.sort_values('avail')

qends = crsp[crsp['date'].dt.month.isin([3,6,9,12]) & crsp['ret'].notna() & (crsp['date']>='1976-10-01')][['permno','date','prc']].copy()
qends = qends.rename(columns={'date':'form_date'})
qends = qends.sort_values('form_date')

formed = pd.merge_asof(
    qends.sort_values('form_date'),
    fv[['permno','avail','FVOL']].sort_values('avail'),
    left_on='form_date', right_on='avail', by='permno', direction='backward'
)
formed = formed.dropna(subset=['FVOL'])
formed = formed[formed['prc'].abs() >= 5].copy()

formed['decile'] = (formed.groupby('form_date')['FVOL']
                    .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1))
formed = formed.dropna(subset=['decile'])
formed['decile'] = formed['decile'].astype(int)
print('formation obs:', len(formed))
print('avg firms/formation:', round(formed.groupby('form_date').size().mean(), 0))

In [ ]:
# 7

formed = formed.sort_values(['permno','form_date']).reset_index(drop=True)

fm = formed[['permno','form_date','decile']].copy()
fm['form_ym'] = pd.to_datetime(fm['form_date']).dt.to_period('M')

holds = []
for k in [1, 2, 3]:
    h = fm[['permno','form_date','decile','form_ym']].copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

ret_panel = crsp[['permno','date','ret_adj','mktcap']].copy()
ret_panel = ret_panel.sort_values(['permno','date'])
ret_panel['w'] = ret_panel.groupby('permno')['mktcap'].shift(1)
ret_panel['ym'] = pd.to_datetime(ret_panel['date']).dt.to_period('M')

m = ret_panel.merge(hold, left_on=['permno','ym'], right_on=['permno','hold_ym'], how='inner')
m = m.dropna(subset=['ret_adj','w'])
m = m.drop_duplicates(subset=['permno','form_date','ym'])

dec_ret = (m.groupby(['ym','decile'])
           .apply(lambda g: (g['ret_adj'] * g['w']).sum() / g['w'].sum(), include_groups=False)
           .reset_index(name='vwret'))

wide = dec_ret.pivot(index='ym', columns='decile', values='vwret')
wide['Low_High'] = wide[1] - wide[10]

lh_vw = wide['Low_High'].dropna()
mean_ret = lh_vw.mean()
t_stat = lh_vw.mean() / (lh_vw.std() / np.sqrt(len(lh_vw)))

print('n months:', len(lh_vw))
print('Low-High monthly mean: {:.4f} ({:.2%})'.format(mean_ret, mean_ret))
print('t-stat: {:.2f}'.format(t_stat))
print(wide[list(range(1,11))].mean().round(4))
print('Low(1): {:.4f}  High(10): {:.4f}'.format(wide[1].mean(), wide[10].mean()))

hold_check = m.groupby(['permno','form_date'])['ym'].nunique()
print('\navg holding months per formation (should be ~3):', round(hold_check.mean(), 2))

In [ ]:
# 8

_qpath = 'q5_factors_monthly_2024.csv'
qf = pd.read_csv(_qpath)
qf['ym'] = pd.PeriodIndex(year=qf['year'], month=qf['month'], freq='M')
for c in ['R_F','R_MKT','R_ME','R_IA','R_ROE','R_EG']:
    qf[c] = qf[c] / 100.0
qf = qf[['ym','R_F','R_MKT','R_ME','R_IA','R_ROE','R_EG']]

_ff5_cache = 'ff5_monthly.parquet'
_mom_cache = 'ff_mom_monthly.parquet'

if os.path.exists(_ff5_cache):
    ff5 = pd.read_parquet(_ff5_cache)
else:
    try: db.get_row_count('ff','fivefactors_monthly')
    except Exception: db = wrds.Connection()
    ff5 = db.get_table('ff','fivefactors_monthly')
    ff5.to_parquet(_ff5_cache)

if os.path.exists(_mom_cache):
    mom = pd.read_parquet(_mom_cache)
else:
    try: db.get_row_count('ff','factors_monthly')
    except Exception: db = wrds.Connection()
    mom = db.get_table('ff','factors_monthly')
    mom.to_parquet(_mom_cache)

print("q-factors:", qf['ym'].min(), "~", qf['ym'].max(), len(qf))
print("ff5 cols:", [c for c in ff5.columns])
print("mom cols:", [c for c in mom.columns])
print("ff5 rows:", len(ff5), " mom rows:", len(mom))
print(ff5.head(2))
print(mom.head(2))

In [ ]:
# 9

ff = ff5.copy()
ff['ym'] = pd.to_datetime(ff['date']).dt.to_period('M')
ff = ff[['ym','mktrf','smb','hml','rmw','cma','umd','rf']]

q = qf.rename(columns={'R_MKT':'qmkt','R_ME':'qme','R_IA':'qia','R_ROE':'qroe'})
q = q[['ym','qmkt','qme','qia','qroe']]
df = lh_vw.rename('lh').reset_index().merge(ff, on='ym', how='left').merge(q, on='ym', how='left')

numcols = ['lh','mktrf','smb','hml','rmw','cma','umd','rf','qmkt','qme','qia','qroe']
for c in numcols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

def alpha(y, X):
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)
    X = sm.add_constant(X)
    m = sm.OLS(y, X).fit()
    return m.params[0], m.tvalues[0]

specs = {
    'raw':  [],
    'CAPM': ['mktrf'],
    'FF3':  ['mktrf','smb','hml'],
    'FF4':  ['mktrf','smb','hml','umd'],
    'FF5':  ['mktrf','smb','hml','rmw','cma'],
    'q4':   ['qmkt','qme','qia','qroe'],
}
paper = {'raw':(0.93,4.09),'CAPM':(1.25,5.89),'FF3':(1.21,6.21),
         'FF4':(1.15,5.85),'FF5':(0.73,4.10),'q4':(0.67,3.45)}

print(f"{'model':6}{'alpha%':>9}{'t':>8}    paper")
for name, cols in specs.items():
    sub = df.dropna(subset=['lh']+cols)
    if name=='raw':
        a = sub['lh'].mean(); t = a/(sub['lh'].std()/np.sqrt(len(sub)))
    else:
        a, t = alpha(sub['lh'], sub[cols])
    pa, pt = paper[name]
    print(f"{name:6}{a*100:9.2f}{t:8.2f}    ({pa:.2f}, t={pt:.2f})")

In [ ]:
# 10

fm=formed[['permno','form_date','decile']].copy()
fm['form_ym']=pd.to_datetime(fm['form_date']).dt.to_period('M')
holds=[]
for k in range(1,25):
    h=fm.copy(); h['hold_ym']=h['form_ym']+k; h['qtr']=(k-1)//3+1; holds.append(h)
hold=pd.concat(holds,ignore_index=True)

rp=crsp[['permno','date','ret_adj','mktcap']].copy().sort_values(['permno','date'])
rp['w']=rp.groupby('permno')['mktcap'].shift(1)
rp['ym']=pd.to_datetime(rp['date']).dt.to_period('M')
mm=rp.merge(hold,left_on=['permno','ym'],right_on=['permno','hold_ym'],how='inner')
mm=mm.dropna(subset=['ret_adj','w']).drop_duplicates(['permno','form_date','ym','qtr'])

dr=(mm.groupby(['qtr','ym','decile'])
    .apply(lambda g:(g['ret_adj']*g['w']).sum()/g['w'].sum(),include_groups=False)
    .reset_index(name='r'))

ffm=ff5.copy(); ffm['ym']=pd.to_datetime(ffm['date']).dt.to_period('M')
ffm=ffm[['ym','mktrf','smb','hml','rmw','cma','umd']]
qmf=qf.rename(columns={'R_MKT':'qmkt','R_ME':'qme','R_IA':'qia','R_ROE':'qroe'})[['ym','qmkt','qme','qia','qroe']]
def al(y,X):
    y=np.asarray(y,float); X=sm.add_constant(np.asarray(X,float)); m=sm.OLS(y,X).fit()
    return m.params[0]*100, m.tvalues[0]
def row(lh):
    d=lh.rename('lh').reset_index().merge(ffm,on='ym',how='left').merge(qmf,on='ym',how='left')
    for c in ['lh','mktrf','smb','hml','rmw','cma','umd','qmkt','qme','qia','qroe']:
        d[c]=pd.to_numeric(d[c],errors='coerce')
    out={}
    r=d['lh'].dropna(); out['Ret']=(r.mean()*100, r.mean()/(r.std()/np.sqrt(len(r))))
    for nm,cols in [('CAPM',['mktrf']),('FF3',['mktrf','smb','hml']),('FF4',['mktrf','smb','hml','umd']),
                    ('FF5',['mktrf','smb','hml','rmw','cma']),('q4',['qmkt','qme','qia','qroe'])]:
        sub=d.dropna(subset=['lh']+cols); out[nm]=al(sub['lh'],sub[cols])
    return out

paper={'Q1':(0.93,1.25,1.21,1.15,0.73,0.67),'Q2':(0.96,1.30,1.25,1.15,0.79,0.71),
       'Q3':(0.88,1.23,1.18,1.07,0.74,0.68),'Q4':(0.83,1.16,1.11,1.00,0.72,0.65),
       'Q5':(0.77,1.08,1.03,0.94,0.67,0.63),'Q6':(0.65,0.93,0.87,0.82,0.55,0.51),
       'Q7':(0.75,1.01,0.97,0.96,0.63,0.60),'Q8':(0.70,0.96,0.93,0.92,0.64,0.65),
       'JT8':(0.81,1.11,1.08,1.02,0.70,0.65)}

print(f"{'Q':4}{'Ret':>7}{'CAPM':>7}{'FF3':>7}{'FF4':>7}{'FF5':>7}{'q4':>7}  (paper Ret/CAPM/FF3/FF4/FF5/q4)")
lh_jt_parts=[]
for qtr in range(1,9):
    w=dr[dr['qtr']==qtr].pivot(index='ym',columns='decile',values='r')
    lh=(w[1]-w[10]).dropna()
    lh_jt_parts.append(lh.rename(f'q{qtr}'))
    o=row(lh)
    p=paper[f'Q{qtr}']
    print(f"Q{qtr:<3}{o['Ret'][0]:7.2f}{o['CAPM'][0]:7.2f}{o['FF3'][0]:7.2f}{o['FF4'][0]:7.2f}{o['FF5'][0]:7.2f}{o['q4'][0]:7.2f}  ({'/'.join(f'{x:.2f}' for x in p)})")

jt=pd.concat(lh_jt_parts,axis=1)
lh_jt=jt.mean(axis=1).dropna()
o=row(lh_jt); p=paper['JT8']
print(f"{'JT8':4}{o['Ret'][0]:7.2f}{o['CAPM'][0]:7.2f}{o['FF3'][0]:7.2f}{o['FF4'][0]:7.2f}{o['FF5'][0]:7.2f}{o['q4'][0]:7.2f}  ({'/'.join(f'{x:.2f}' for x in p)})")

In [ ]:
# 11

DSF_FILE = 'crsp_dsf_raw.parquet'
FFD_FILE = 'ff_factors_daily.parquet'

if os.path.exists(DSF_FILE):
    dsf = pd.read_parquet(DSF_FILE)
else:
    q = f"""
        SELECT permno, date, ret
        FROM crsp.dsf
        WHERE date >= '{START_DATE}' AND date <= '2024-12-31'
    """
    try: db.get_row_count('crsp','dsf')
    except Exception: db = wrds.Connection()
    dsf = db.raw_sql(q, date_cols=['date'])
    dsf['permno'] = dsf['permno'].astype('int64')
    dsf.to_parquet(DSF_FILE)

if os.path.exists(FFD_FILE):
    ffd = pd.read_parquet(FFD_FILE)
else:
    try: db.get_row_count('ff','factors_daily')
    except Exception: db = wrds.Connection()
    ffd = db.get_table('ff','factors_daily')
    ffd.to_parquet(FFD_FILE)

print('dsf:', dsf.shape)
print('ffd:', ffd.shape, ffd.columns.tolist())
print('dsf date:', dsf['date'].min(), '~', dsf['date'].max())

In [ ]:
# 12

if os.path.exists('ivol_quarterly.parquet'):
    ivol = pd.read_parquet('ivol_quarterly.parquet')
    print('ivol loaded from cache:', len(ivol))
    print('ivol per qtr:', round(ivol.groupby('qtr').size().mean(), 0))
    print(ivol['ivol'].describe().round(4))
else:
    f = ffd[['date','mktrf','smb','hml','rf']].copy()
    f['date'] = pd.to_datetime(f['date'])
    for c in ['mktrf','smb','hml','rf']:
        f[c] = pd.to_numeric(f[c], errors='coerce').astype('float32')

    d = dsf[['permno','date','ret']].copy()
    d['ret'] = pd.to_numeric(d['ret'], errors='coerce').astype('float32')
    d = d.dropna(subset=['ret'])
    d['date'] = pd.to_datetime(d['date'])

    d = d.merge(f, on='date', how='inner')
    del f

    permno = d['permno'].values.astype('int64')
    qtr_ord = pd.PeriodIndex(d['date'].dt.to_period('Q')).asi8
    y = (d['ret'].values - d['rf'].values).astype('float32')
    mkt = d['mktrf'].values.astype('float32')
    smb = d['smb'].values.astype('float32')
    hml = d['hml'].values.astype('float32')
    del d

    key = permno.astype('int64') * 100000 + qtr_ord.astype('int64')
    uniq, gid = np.unique(key, return_inverse=True)
    G = len(uniq)
    del key

    ones = np.ones(len(y), dtype='float32')
    cols = [ones, mkt, smb, hml]
    k = 4
    XtX = np.zeros((G, k, k), dtype='float64')
    Xty = np.zeros((G, k), dtype='float64')
    for i in range(k):
        Xty[:, i] = np.bincount(gid, weights=(cols[i]*y).astype('float64'), minlength=G)
        for j in range(i, k):
            v = np.bincount(gid, weights=(cols[i]*cols[j]).astype('float64'), minlength=G)
            XtX[:, i, j] = v
            XtX[:, j, i] = v

    cnt = np.bincount(gid, minlength=G)
    syy = np.bincount(gid, weights=(y.astype('float64'))**2, minlength=G)

    ivol_arr = np.full(G, np.nan)
    for g in range(G):
        if cnt[g] < 15:
            continue
        try:
            b = np.linalg.solve(XtX[g], Xty[g])
            ssr = syy[g] - b @ Xty[g]
            if ssr > 0:
                ivol_arr[g] = np.sqrt(ssr / (cnt[g] - k))
        except np.linalg.LinAlgError:
            pass

    up = (uniq // 100000).astype('int64')
    uord = (uniq % 100000).astype('int64')
    ivol = pd.DataFrame({'permno': up, 'qtr_ord': uord, 'ivol': ivol_arr}).dropna()
    ivol['qtr'] = ivol['qtr_ord'].map(lambda o: pd.Period(ordinal=int(o), freq='Q'))
    ivol = ivol[['permno','qtr','ivol']]

    print('ivol rows:', len(ivol))
    print('ivol per qtr:', round(ivol.groupby('qtr').size().mean(), 0))
    print(ivol['ivol'].describe().round(4))
    ivol.to_parquet('ivol_quarterly.parquet')

In [ ]:
# 13

iv = ivol.copy()
iv['permno'] = iv['permno'].astype('int64')

fm = formed[['permno','form_date','decile','FVOL']].copy()
fm['form_qtr'] = pd.to_datetime(fm['form_date']).dt.to_period('Q')
fm['prev_qtr'] = fm['form_qtr'] - 1
fm = fm.merge(iv.rename(columns={'qtr':'prev_qtr'}), on=['permno','prev_qtr'], how='left')
fm = fm.dropna(subset=['ivol'])

fm['fq'] = fm.groupby('form_date')['FVOL'].transform(lambda x: pd.qcut(x,5,labels=False,duplicates='drop')+1)
fm['iq'] = fm.groupby('form_date')['ivol'].transform(lambda x: pd.qcut(x,5,labels=False,duplicates='drop')+1)
fm = fm.dropna(subset=['fq','iq'])
fm['fq'] = fm['fq'].astype(int)
fm['iq'] = fm['iq'].astype(int)
fm['form_ym'] = pd.to_datetime(fm['form_date']).dt.to_period('M')

print('dual-sort firms/form:', round(fm.groupby('form_date').size().mean(),0))
print('ivol attach rate:', round(len(fm)/len(formed)*100,1))

holds = []
for k in range(1,25):
    h = fm[['permno','form_ym','fq','iq']].copy()
    h['hold_ym'] = h['form_ym']+k
    h['qt'] = (k-1)//3+1
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

rp = crsp[['permno','date','ret_adj','mktcap']].copy().sort_values(['permno','date'])
rp['w'] = rp.groupby('permno')['mktcap'].shift(1)
rp['ym'] = pd.to_datetime(rp['date']).dt.to_period('M')
mm = rp.merge(hold, left_on=['permno','ym'], right_on=['permno','hold_ym'], how='inner')
mm = mm.dropna(subset=['ret_adj','w']).drop_duplicates(['permno','form_ym','ym','qt'])

cell = (mm.groupby(['qt','ym','iq','fq'])
        .apply(lambda g:(g['ret_adj']*g['w']).sum()/g['w'].sum(), include_groups=False)
        .reset_index(name='r'))

ffm = ff5.copy()
ffm['ym'] = pd.to_datetime(ffm['date']).dt.to_period('M')
ffm = ffm[['ym','mktrf','smb','hml','rmw','cma']]
qmf = qf.rename(columns={'R_MKT':'qmkt','R_ME':'qme','R_IA':'qia','R_ROE':'qroe'})[['ym','qmkt','qme','qia','qroe']]

def al(yv, X):
    yv = np.asarray(yv,float)
    X = sm.add_constant(np.asarray(X,float))
    m = sm.OLS(yv,X).fit()
    return m.params[0]*100, m.tvalues[0]

def stat(lh):
    d = lh.rename('lh').reset_index().merge(ffm,on='ym',how='left').merge(qmf,on='ym',how='left')
    for c in ['lh','mktrf','smb','hml','rmw','cma','qmkt','qme','qia','qroe']:
        d[c] = pd.to_numeric(d[c],errors='coerce')
    r = d['lh'].dropna()
    ret = (r.mean()*100, r.mean()/(r.std()/np.sqrt(len(r))))
    a5 = d.dropna(subset=['lh','mktrf','smb','hml','rmw','cma'])
    s5 = al(a5['lh'], a5[['mktrf','smb','hml','rmw','cma']])
    aq = d.dropna(subset=['lh','qmkt','qme','qia','qroe'])
    sq = al(aq['lh'], aq[['qmkt','qme','qia','qroe']])
    return ret, s5, sq

paper = {1:(0.52,0.37,0.36),2:(0.61,0.46,0.46),3:(0.69,0.57,0.58),4:(0.63,0.53,0.55),
         5:(0.46,0.32,0.32),6:(0.48,0.35,0.36),7:(0.49,0.36,0.35),8:(0.44,0.33,0.30),'JT8':(0.53,0.41,0.41)}

print(f"{'Q':5}{'Ret':>7}{'t':>6}{'FF5':>7}{'q4':>7}   paper(Ret/FF5/q4)")
parts = []
for qt in range(1,9):
    sub = cell[cell['qt']==qt]
    lhs = []
    for iqv in range(1,6):
        w = sub[sub['iq']==iqv].pivot(index='ym',columns='fq',values='r')
        if 1 in w and 5 in w:
            lhs.append((w[1]-w[5]).rename(iqv))
    lh = pd.concat(lhs,axis=1).mean(axis=1).dropna()
    parts.append(lh)
    ret,s5,sq = stat(lh)
    p = paper[qt]
    print(f"Q{qt:<4}{ret[0]:7.2f}{ret[1]:6.2f}{s5[0]:7.2f}{sq[0]:7.2f}   ({p[0]}/{p[1]}/{p[2]})")
jt = pd.concat(parts,axis=1).mean(axis=1).dropna()
ret,s5,sq = stat(jt)
p = paper['JT8']
print(f"{'JT8':5}{ret[0]:7.2f}{ret[1]:6.2f}{s5[0]:7.2f}{sq[0]:7.2f}   ({p[0]}/{p[1]}/{p[2]})")

In [ ]:
# 14

paperB = {1:(0.26,0.41,0.36),2:(0.12,0.36,0.36),3:(-0.08,0.17,0.17),4:(-0.09,0.17,0.17),
          5:(-0.06,0.20,0.20),6:(-0.11,0.14,0.14),7:(-0.12,0.09,0.09),8:(-0.20,0.05,0.05),'JT8':(-0.06,0.20,0.20)}

print(f"{'Q':5}{'Ret':>7}{'t':>6}{'FF5':>7}{'q4':>7}   paper(Ret/FF5/q4)")
partsB = []
for qt in range(1,9):
    sub = cell[cell['qt']==qt]
    lhs = []
    for fqv in range(1,6):
        w = sub[sub['fq']==fqv].pivot(index='ym',columns='iq',values='r')
        if 1 in w and 5 in w:
            lhs.append((w[1]-w[5]).rename(fqv))
    lh = pd.concat(lhs,axis=1).mean(axis=1).dropna()
    partsB.append(lh)
    ret,s5,sq = stat(lh)
    p = paperB[qt]
    print(f"Q{qt:<4}{ret[0]:7.2f}{ret[1]:6.2f}{s5[0]:7.2f}{sq[0]:7.2f}   ({p[0]}/{p[1]}/{p[2]})")
jt = pd.concat(partsB,axis=1).mean(axis=1).dropna()
ret,s5,sq = stat(jt)
p = paperB['JT8']
print(f"{'JT8':5}{ret[0]:7.2f}{ret[1]:6.2f}{s5[0]:7.2f}{sq[0]:7.2f}   ({p[0]}/{p[1]}/{p[2]})")

In [ ]:
# 15

PROF_FILE = 'comp_prof_raw.parquet'

if os.path.exists(PROF_FILE):
    prof = pd.read_parquet(PROF_FILE)
else:
    pcols = ['gvkey','datadate','revtq','cogsq','xsgaq','xintq','ibq','atq','seqq','txditcq','pstkq','pstkrq']
    q = f"""
        SELECT {', '.join(pcols)}
        FROM comp.fundq
        WHERE indfmt='INDL' AND datafmt='STD' AND consol='C' AND popsrc='D'
          AND datadate >= '{START_DATE}'
    """
    try: db.get_row_count('comp','fundq')
    except Exception: db = wrds.Connection()
    prof = db.raw_sql(q, date_cols=['datadate'])
    prof = prof.drop_duplicates(subset=['gvkey','datadate'])
    prof.to_parquet(PROF_FILE)

print('prof:', prof.shape)
print(prof[['revtq','cogsq','xsgaq','xintq','ibq','seqq','txditcq','pstkq','pstkrq']].notna().mean().round(3).to_dict())

In [ ]:
# 16

p = prof.copy()
p = p.sort_values(['gvkey','datadate']).reset_index(drop=True)

pref = p['pstkrq'].where(p['pstkrq'].notna(), p['pstkq'])
p['qbe'] = p['seqq'] + p['txditcq'].fillna(0) - pref.fillna(0)

g = p.groupby('gvkey', sort=False)
p['atq_lag'] = g['atq'].shift(1)
p['qbe_lag'] = g['qbe'].shift(1)

p['gp'] = (p['revtq'] - p['cogsq']) / p['atq_lag'].where(p['atq_lag'] > 0)
p['op'] = (p['revtq'] - p['cogsq'] - p['xsgaq'].fillna(0) - p['xintq'].fillna(0)) / p['qbe_lag'].where(p['qbe_lag'] > 0)
p['roe'] = p['ibq'] / p['qbe_lag'].where(p['qbe_lag'] > 0)

prof_calc = p[['gvkey','datadate','gp','op','roe']].copy()

print('gp/op/roe notna:', prof_calc[['gp','op','roe']].notna().mean().round(3).to_dict())
print(prof_calc[['gp','op','roe']].describe(percentiles=[.05,.5,.95]).round(3))
prof_calc.to_parquet('prof_calc.parquet')

In [ ]:
# 17

pc = prof_calc.copy()
fm = formed[['permno','form_date','decile']].copy()

l2 = link[['gvkey','permno','linkdt','linkenddt']].copy()
l2['permno'] = l2['permno'].astype('int64')
pc2 = pc.merge(l2, on='gvkey', how='inner')
pc2 = pc2[(pc2['datadate']>=pc2['linkdt']) & (pc2['datadate']<=pc2['linkenddt'])]
pc2['permno'] = pc2['permno'].astype('int64')
pc2['form_date'] = pd.to_datetime(pc2['datadate'])

fm['form_date'] = pd.to_datetime(fm['form_date'])
fm = fm.sort_values('form_date')
pc2 = pc2.sort_values('form_date')

m = pd.merge_asof(fm, pc2[['permno','form_date','gp','op','roe']].sort_values('form_date'),
                  on='form_date', by='permno', direction='backward',
                  tolerance=pd.Timedelta('120D'))

def w(s):
    return s.clip(s.quantile(0.01), s.quantile(0.99))
for c in ['gp','op','roe']:
    m[c] = m.groupby('form_date')[c].transform(w)

t = m.groupby('decile')[['gp','op','roe']].mean()*100
print(f"{'dec':4}{'GP%':>8}{'OP%':>8}{'ROE%':>8}")
for d in range(1,11):
    print(f"{d:<4}{t.loc[d,'gp']:8.1f}{t.loc[d,'op']:8.1f}{t.loc[d,'roe']:8.1f}")
print("paper: GP 14->4, OP 8->-0.1, ROE 4->-3")

In [ ]:
# 18

be = prof[['gvkey','datadate','seqq','txditcq','pstkrq','pstkq']].copy()
pref = be['pstkrq'].where(be['pstkrq'].notna(), be['pstkq'])
be['be'] = be['seqq'] + be['txditcq'].fillna(0) - pref.fillna(0)
be['ym'] = pd.to_datetime(be['datadate']).dt.to_period('M')
l2 = link[['gvkey','permno','linkdt','linkenddt']].copy()
l2['permno'] = l2['permno'].astype('int64')
be = be.merge(l2, on='gvkey', how='inner')
be = be[(be['datadate']>=be['linkdt']) & (be['datadate']<=be['linkenddt'])]
be['permno'] = be['permno'].astype('int64')
me = crsp[['permno','date','mktcap']].copy()
me['ym'] = pd.to_datetime(me['date']).dt.to_period('M')
me = me.drop_duplicates(['permno','ym'])
be = be.merge(me[['permno','ym','mktcap']], on=['permno','ym'], how='left')
be['bm'] = be['be'].where(be['be']>0) / (be['mktcap'].where(be['mktcap']>0) / 1000)
bm = be[['permno','datadate','bm']].dropna()

In [ ]:
# 19

FUNDA_FILE = 'comp_funda_raw.parquet'

if os.path.exists(FUNDA_FILE):
    funda = pd.read_parquet(FUNDA_FILE)
else:
    fcols = ['gvkey','datadate','fyear','at','lt','ni','act','lct','che','dlc','txp',
             'dp','dvt','ib','oancf','ceq','sale','cogs','dltt','re','ppegt','invt','rect']
    q = f"""
        SELECT {', '.join(fcols)}
        FROM comp.funda
        WHERE indfmt='INDL' AND datafmt='STD' AND consol='C' AND popsrc='D'
          AND datadate >= '{START_DATE}'
    """
    try: db.get_row_count('comp','funda')
    except Exception: db = wrds.Connection()
    funda = db.raw_sql(q, date_cols=['datadate'])
    funda = funda.drop_duplicates(subset=['gvkey','datadate'])
    funda.to_parquet(FUNDA_FILE)

print('funda:', funda.shape)
print(funda[['at','lt','ni','act','lct','dvt','ib','oancf','dltt']].notna().mean().round(3).to_dict())

In [ ]:
# 20

fa = funda.sort_values(['gvkey','datadate']).reset_index(drop=True)
g = fa.groupby('gvkey', sort=False)

fa['at_lag'] = g['at'].shift(1)
fa['act_lag'] = g['act'].shift(1)
fa['lct_lag'] = g['lct'].shift(1)
fa['che_lag'] = g['che'].shift(1)
fa['dlc_lag'] = g['dlc'].shift(1)
fa['txp_lag'] = g['txp'].shift(1)

d_act = fa['act'] - fa['act_lag']
d_che = fa['che'] - fa['che_lag']
d_lct = fa['lct'] - fa['lct_lag']
d_dlc = fa['dlc'] - fa['dlc_lag']
d_txp = fa['txp'] - fa['txp_lag']
fa['accruals'] = (d_act - d_che) - (d_lct - d_dlc - d_txp.fillna(0)) - fa['dp']

fa['be'] = fa['ceq'].where(fa['ceq'].notna(), fa['at'] - fa['lt'])
fa['acc_be'] = fa['accruals'] / fa['be'].where(fa['be']>0)
fa['neg_accbe'] = fa['acc_be'].where(fa['acc_be']<0, 0)
fa['pos_accbe'] = fa['acc_be'].where(fa['acc_be']>0, 0)

fa['ag'] = (fa['at'] - fa['at_lag']) / fa['at_lag'].where(fa['at_lag']>0)

fa['nodiv'] = (fa['dvt'].fillna(0) <= 0).astype(float)
fa['divbe'] = fa['dvt'].fillna(0) / fa['be'].where(fa['be']>0)

ctrl_annual = fa[['gvkey','datadate','fyear','neg_accbe','pos_accbe','ag','nodiv','divbe']].copy()
print('ctrl_annual:', ctrl_annual.shape)
print(ctrl_annual[['neg_accbe','pos_accbe','ag','nodiv','divbe']].describe(percentiles=[.05,.5,.95]).round(3))

In [ ]:
# 21

fo = funda.sort_values(['gvkey','datadate']).reset_index(drop=True)
g = fo.groupby('gvkey', sort=False)
fo['ni_lag'] = g['ni'].shift(1)

TA = fo['at'].where(fo['at']>0)
TL = fo['lt']
WC = fo['act'] - fo['lct']
CL = fo['lct']
CA = fo['act']

oscore = (-1.32
          - 0.407*np.log(TA)
          + 6.03*(TL/TA)
          - 1.43*(WC/TA)
          + 0.0757*(CL/CA.where(CA>0))
          - 1.72*(TL > TA).astype(float)
          - 2.37*(fo['ni']/TA)
          - 1.83*(fo['oancf']/TL.where(TL>0))
          + 0.285*((fo['ni']<0) & (fo['ni_lag']<0)).astype(float)
          - 0.521*((fo['ni']-fo['ni_lag'])/(fo['ni'].abs()+fo['ni_lag'].abs()).where((fo['ni'].abs()+fo['ni_lag'].abs())>0)))

fo['oscore'] = oscore
oscore_df = fo[['gvkey','datadate','fyear','oscore']].dropna(subset=['oscore'])
print('oscore:', oscore_df.shape)
print(oscore_df['oscore'].describe(percentiles=[.05,.5,.95]).round(3))

In [ ]:
# 22

ff_ = funda.sort_values(['gvkey','datadate']).reset_index(drop=True)
g = ff_.groupby('gvkey', sort=False)

ff_['at_lag'] = g['at'].shift(1)
ff_['roa'] = ff_['ni'] / ff_['at_lag'].where(ff_['at_lag']>0)
ff_['roa_lag'] = g['roa'].shift(1)
ff_['cfo'] = ff_['oancf'] / ff_['at_lag'].where(ff_['at_lag']>0)

ff_['lev'] = ff_['dltt'] / ff_['at']
ff_['lev_lag'] = g['lev'].shift(1)
ff_['cur'] = ff_['act'] / ff_['lct'].where(ff_['lct']>0)
ff_['cur_lag'] = g['cur'].shift(1)

ff_['margin'] = (ff_['sale'] - ff_['cogs']) / ff_['sale'].where(ff_['sale']>0)
ff_['margin_lag'] = g['margin'].shift(1)
ff_['turn'] = ff_['sale'] / ff_['at_lag'].where(ff_['at_lag']>0)
ff_['turn_lag'] = g['turn'].shift(1)

ceq_now = ff_['ceq']
ceq_lag = g['ceq'].shift(1)
issue = (ceq_now - ceq_lag)

f1 = (ff_['roa'] > 0).astype(float)
f2 = (ff_['cfo'] > 0).astype(float)
f3 = (ff_['roa'] > ff_['roa_lag']).astype(float)
f4 = (ff_['cfo'] > ff_['roa']).astype(float)
f5 = (ff_['lev'] < ff_['lev_lag']).astype(float)
f6 = (ff_['cur'] > ff_['cur_lag']).astype(float)
f7 = (issue <= 0).astype(float)
f8 = (ff_['margin'] > ff_['margin_lag']).astype(float)
f9 = (ff_['turn'] > ff_['turn_lag']).astype(float)

ff_['fscore'] = f1+f2+f3+f4+f5+f6+f7+f8+f9
fscore_df = ff_[['gvkey','datadate','fyear','fscore']].dropna(subset=['fscore'])
print('fscore:', fscore_df.shape)
print(fscore_df['fscore'].value_counts().sort_index().to_dict())

In [ ]:
# 23

mr = crsp[['permno','date','ret']].copy()
mr['ret'] = pd.to_numeric(mr['ret'], errors='coerce')
mr = mr.dropna(subset=['ret']).sort_values(['permno','date']).reset_index(drop=True)
mr['lg'] = np.log1p(mr['ret'])
mr['ym'] = pd.to_datetime(mr['date']).dt.to_period('M')

g = mr.groupby('permno', sort=False)['lg']
r12 = g.rolling(12, min_periods=10).sum().reset_index(level=0, drop=True)
r36 = g.rolling(36, min_periods=24).sum().reset_index(level=0, drop=True)
r12_lag = mr.groupby('permno')['lg'].shift(0)

mr['ret_1yr'] = np.expm1(r12)
cum36 = np.expm1(r36)
cum12 = np.expm1(r12)
mr['ret_23yr'] = (1 + cum36) / (1 + cum12) - 1

mr['meas_qtr'] = mr['ym'].dt.asfreq('Q')
past_ret = (mr.sort_values(['permno','date'])
            .groupby(['permno','meas_qtr'])
            .agg(ret_1yr=('ret_1yr','last'), ret_23yr=('ret_23yr','last'))
            .reset_index())

print('past_ret:', past_ret.shape)
print(past_ret[['ret_1yr','ret_23yr']].describe(percentiles=[.05,.5,.95]).round(3))

In [ ]:
# 24

cphase = prof_calc.copy()
cphase['meas_qtr'] = pd.to_datetime(cphase['datadate']).dt.to_period('Q')
l2 = link[['gvkey','permno','linkdt','linkenddt']].copy()
l2['permno'] = l2['permno'].astype('int64')
cphase = cphase.merge(l2, on='gvkey', how='inner')
cphase = cphase[(cphase['datadate']>=cphase['linkdt']) & (cphase['datadate']<=cphase['linkenddt'])]
cphase['permno'] = cphase['permno'].astype('int64')
prof_q = cphase[['permno','meas_qtr','gp','op','roe']].copy()

pp = prof_calc.copy()
pp['meas_qtr'] = pd.to_datetime(pp['datadate']).dt.to_period('Q')
pp = pp.merge(l2, on='gvkey', how='inner')
pp = pp[(pp['datadate']>=pp['linkdt']) & (pp['datadate']<=pp['linkenddt'])]
pp['permno'] = pp['permno'].astype('int64')
pp = pp[['permno','meas_qtr','roe']].rename(columns={'roe':'past_roe'})
pp['np'] = (pp['past_roe'] < 0).astype(float)

base5 = formed[['permno','form_date','FVOL']].copy()
base5['FVOL'] = base5['FVOL']*100
base5['meas_qtr'] = pd.to_datetime(base5['form_date']).dt.to_period('Q') - 1

iv = ivol.rename(columns={'qtr':'meas_qtr'}).copy()
iv['permno'] = iv['permno'].astype('int64')

bm5 = bm.copy()
bm5['meas_qtr'] = pd.to_datetime(bm5['datadate']).dt.to_period('Q')
bm5 = bm5[['permno','meas_qtr','bm']]

me5 = crsp[['permno','date','mktcap']].copy()
me5['meas_qtr'] = pd.to_datetime(me5['date']).dt.to_period('Q')
me5 = me5.groupby(['permno','meas_qtr'])['mktcap'].last().reset_index()
me5['size'] = np.log(me5['mktcap'].where(me5['mktcap']>0))
me5 = me5[['permno','meas_qtr','size']]

def to_q(df, vcols):
    x = df.copy()
    x['fy'] = pd.to_datetime(x['datadate']).dt.year
    x = x.merge(l2, on='gvkey', how='inner')
    x = x[(x['datadate']>=x['linkdt']) & (x['datadate']<=x['linkenddt'])]
    x['permno'] = x['permno'].astype('int64')
    out = []
    for yq in range(4):
        t = x[['permno','fy']+vcols].copy()
        t['meas_qtr'] = pd.PeriodIndex(year=t['fy']+1, quarter=yq+1, freq='Q')
        out.append(t[['permno','meas_qtr']+vcols])
    return pd.concat(out, ignore_index=True).drop_duplicates(['permno','meas_qtr'])

acc_q = to_q(ctrl_annual, ['neg_accbe','pos_accbe','ag','nodiv','divbe'])
os_q = to_q(oscore_df, ['oscore'])
fs_q = to_q(fscore_df, ['fscore'])

pr = past_ret.copy()
pr['permno'] = pr['permno'].astype('int64')

panel = base5.merge(iv[['permno','meas_qtr','ivol']], on=['permno','meas_qtr'], how='left')
panel = panel.merge(bm5, on=['permno','meas_qtr'], how='left')
panel = panel.merge(me5, on=['permno','meas_qtr'], how='left')
panel = panel.merge(pp[['permno','meas_qtr','past_roe','np']], on=['permno','meas_qtr'], how='left')
panel = panel.merge(acc_q, on=['permno','meas_qtr'], how='left')
panel = panel.merge(os_q, on=['permno','meas_qtr'], how='left')
panel = panel.merge(fs_q, on=['permno','meas_qtr'], how='left')
panel = panel.merge(pr[['permno','meas_qtr','ret_1yr','ret_23yr']], on=['permno','meas_qtr'], how='left')

ctrls = ['ivol','bm','size','past_roe','np','neg_accbe','pos_accbe','ag','nodiv','divbe','oscore','fscore','ret_1yr','ret_23yr']

def winz(s): return s.clip(s.quantile(.01), s.quantile(.99))
def stdz(s):
    sd = s.std()
    return (s - s.mean())/sd if sd and sd>0 else s*0

results = {}
for prof_name in ['gp','op','roe']:
    rows_uni, rows_multi = [], []
    for h in range(1,9):
        tgt = prof_q[['permno','meas_qtr',prof_name]].copy()
        tgt['form_q'] = tgt['meas_qtr'] - h
        d = panel.merge(tgt[['permno','form_q',prof_name]],
                        left_on=['permno','meas_qtr'], right_on=['permno','form_q'], how='inner')
        d = d.dropna(subset=[prof_name,'FVOL'])
        d['form_q'] = d['meas_qtr']
        d['y'] = d.groupby('form_q')[prof_name].transform(winz)*100

        bu = []
        for q, gg in d.groupby('form_q'):
            if len(gg) < 30: continue
            X = sm.add_constant(gg[['FVOL']].values.astype(float))
            bu.append(sm.OLS(gg['y'].values.astype(float), X).fit().params[1])
        rows_uni.append((h, np.mean(bu), np.mean(bu)/(np.std(bu)/np.sqrt(len(bu)))))

        dm = d.dropna(subset=ctrls).copy()
        for c in ctrls:
            dm[c] = dm.groupby('form_q')[c].transform(winz)
            dm[c] = dm.groupby('form_q')[c].transform(stdz)
        bmv = []
        for q, gg in dm.groupby('form_q'):
            if len(gg) < 50: continue
            X = sm.add_constant(gg[['FVOL']+ctrls].values.astype(float))
            try:
                bmv.append(sm.OLS(gg['y'].values.astype(float), X).fit().params[1])
            except Exception:
                pass
        rows_multi.append((h, np.mean(bmv), np.mean(bmv)/(np.std(bmv)/np.sqrt(len(bmv))) if len(bmv)>1 else np.nan))
    results[prof_name] = (rows_uni, rows_multi)

for pn in ['gp','op','roe']:
    uni, multi = results[pn]
    print(f"\n{pn.upper()}")
    print(f"{'Q':4}{'uni_b':>9}{'uni_t':>8}{'multi_b':>10}{'multi_t':>9}")
    for (h,bu,tu),(_,bmval,tmv) in zip(uni,multi):
        print(f"Q{h:<3}{bu:9.3f}{tu:8.2f}{bmval:10.3f}{tmv:9.2f}")

In [ ]:
# 25

l2 = link[['gvkey','permno','linkdt','linkenddt']].copy()
l2['permno'] = l2['permno'].astype('int64')

cphase = prof_calc.copy()
cphase['hold_qtr'] = pd.to_datetime(cphase['datadate']).dt.to_period('Q')
cphase = cphase.merge(l2, on='gvkey', how='inner')
cphase = cphase[(cphase['datadate']>=cphase['linkdt']) & (cphase['datadate']<=cphase['linkenddt'])]
cphase['permno'] = cphase['permno'].astype('int64')
prof_hold = cphase[['permno','hold_qtr','gp','op','roe']].copy()

base6 = formed[['permno','form_date','FVOL']].copy()
base6['FVOL'] = base6['FVOL']*100
base6['form_qtr'] = pd.to_datetime(base6['form_date']).dt.to_period('Q')
base6['hold_qtr'] = base6['form_qtr'] + 1

qr = crsp[['permno','date','ret']].copy()
qr['ret'] = pd.to_numeric(qr['ret'], errors='coerce')
ffm = ff5.copy()
ffm['ym'] = pd.to_datetime(ffm['date']).dt.to_period('M')
ffm['rf'] = pd.to_numeric(ffm['rf'], errors='coerce')
qr['ym'] = pd.to_datetime(qr['date']).dt.to_period('M')
qr = qr.merge(ffm[['ym','rf']], on='ym', how='left')
qr['exret'] = qr['ret'] - qr['rf']
qr['hold_qtr'] = qr['ym'].dt.asfreq('Q')
qq = qr.groupby(['permno','hold_qtr'])['exret'].apply(lambda s:(1+s).prod()-1).reset_index(name='qret')

d = base6.merge(qq, on=['permno','hold_qtr'], how='inner')
d = d.merge(prof_hold, on=['permno','hold_qtr'], how='left')
d = d.dropna(subset=['qret','FVOL'])
d['y'] = d.groupby('form_qtr')['qret'].transform(lambda s: s.clip(s.quantile(.01), s.quantile(.99)))*100

def winz(s): return s.clip(s.quantile(.01), s.quantile(.99))
def stdz(s):
    sd = s.std()
    return (s-s.mean())/sd if sd and sd>0 else s*0

specs = {'(1) FVOL only': [], '(3) +GP': ['gp'], '(4) +OP': ['op'], '(5) +ROE': ['roe']}
print(f"{'spec':14}{'FVOL_b':>10}{'t':>8}")
for name, extra in specs.items():
    dd = (d.dropna(subset=extra) if extra else d).copy()
    for c in extra:
        dd[c] = dd.groupby('form_qtr')[c].transform(winz)
        dd[c] = dd.groupby('form_qtr')[c].transform(stdz)
    coefs = []
    for q, g in dd.groupby('form_qtr'):
        if len(g) < 30: continue
        X = sm.add_constant(g[['FVOL']+extra].values.astype(float))
        try: coefs.append(sm.OLS(g['y'].values.astype(float), X).fit().params[1])
        except: pass
    b = np.mean(coefs); t = b/(np.std(coefs)/np.sqrt(len(coefs)))
    print(f"{name:14}{b:10.4f}{t:8.2f}")

In [ ]:
# 26

fm = formed[['permno','form_date','decile']].copy()
fm['form_ym'] = pd.to_datetime(fm['form_date']).dt.to_period('M')

holds = []
for k in [1, 2, 3]:
    h = fm.copy()
    h['hold_ym'] = h['form_ym'] + k
    holds.append(h)
hold = pd.concat(holds, ignore_index=True)

rp = crsp[['permno','date','ret_adj']].copy().sort_values(['permno','date'])
rp['ym'] = pd.to_datetime(rp['date']).dt.to_period('M')
mm = rp.merge(hold, left_on=['permno','ym'], right_on=['permno','hold_ym'], how='inner')
mm = mm.dropna(subset=['ret_adj']).drop_duplicates(['permno','form_date','ym'])

ew = mm.groupby(['ym','decile'])['ret_adj'].mean().reset_index(name='ewret')
wide = ew.pivot(index='ym', columns='decile', values='ewret')
lh_ew = (wide[1] - wide[10]).dropna()

print('EW Low-High: {:.2f}% t={:.2f}'.format(lh_ew.mean()*100, lh_ew.mean()/(lh_ew.std()/np.sqrt(len(lh_ew)))))
print('EW Low(1): {:.2f}%  High(10): {:.2f}%'.format(wide[1].mean()*100, wide[10].mean()*100))
print('(VW 0.65%, paper VW 0.93%)')

In [ ]:
# 27

fvf = lh_vw.rename('fvol').reset_index()

ffm = ff5.copy()
ffm['ym'] = pd.to_datetime(ffm['date']).dt.to_period('M')
ffm = ffm[['ym','mktrf','smb','hml','rmw','cma']]
qmf = qf.rename(columns={'R_MKT':'qmkt','R_ME':'qme','R_IA':'qia','R_ROE':'qroe'})[['ym','qmkt','qme','qia','qroe']]

d = fvf.merge(ffm, on='ym', how='left').merge(qmf, on='ym', how='left')
for c in d.columns:
    if c != 'ym': d[c] = pd.to_numeric(d[c], errors='coerce')

def sharpe2(R):
    R = np.asarray(R, float)
    mu = R.mean(axis=0)
    cov = np.cov(R, rowvar=False)
    return mu @ np.linalg.solve(cov, mu)

def spanning(factor_col, base_cols, label):
    sub = d.dropna(subset=[factor_col]+base_cols)
    y = sub[factor_col].to_numpy(dtype=float)
    X = sm.add_constant(sub[base_cols].to_numpy(dtype=float))
    m = sm.OLS(y, X).fit()
    a, ta = m.params[0]*100, m.tvalues[0]
    rbar = sub[factor_col].mean()*100
    tr = sub[factor_col].mean()/(sub[factor_col].std()/np.sqrt(len(sub)))
    sr0 = sharpe2(sub[base_cols].to_numpy(dtype=float))
    sr1 = sharpe2(sub[[factor_col]+base_cols].to_numpy(dtype=float))
    print(f'{label}: rbar {rbar:.2f}%(t{tr:.2f})  alpha {a:.2f}%(t{ta:.2f})  SR2_0 {sr0:.3f} SR2 {sr1:.3f}  IR2 {sr1-sr0:.3f}')

print('=== Panel A: vs FF5 ===  (paper: alpha 0.89% t6.29, SR2 0.11->0.18)')
spanning('fvol', ['mktrf','smb','hml','rmw','cma'], 'FVOL')
print('\n=== Panel B: vs q4 ===  (paper: alpha 0.86% t5.82, SR2 0.14->0.21)')
spanning('fvol', ['qmkt','qme','qia','qroe'], 'FVOL')

In [ ]:
# 28

_FF30_RANGES = [
(100,199,1),(200,299,1),(700,799,1),(910,919,1),(2000,2009,1),(2010,2019,1),(2020,2029,1),(2030,2039,1),(2040,2046,1),(2048,2048,1),(2050,2059,1),(2060,2063,1),(2064,2068,1),(2070,2079,1),(2086,2086,1),(2087,2087,1),(2090,2092,1),(2095,2095,1),(2096,2096,1),(2097,2097,1),(2098,2099,1),
(2080,2080,2),(2082,2082,2),(2083,2083,2),(2084,2084,2),(2085,2085,2),
(2100,2199,3),
(920,999,4),(3650,3651,4),(3652,3652,4),(3732,3732,4),(3930,3931,4),(3940,3949,4),(7800,7829,4),(7830,7833,4),(7840,7841,4),(7900,7900,4),(7910,7911,4),(7920,7929,4),(7930,7933,4),(7940,7949,4),(7980,7980,4),(7990,7999,4),
(2700,2709,5),(2710,2719,5),(2720,2729,5),(2730,2739,5),(2740,2749,5),(2750,2759,5),(2770,2771,5),(2780,2789,5),(2790,2799,5),(3993,3993,5),
(2047,2047,6),(2391,2392,6),(2510,2519,6),(2590,2599,6),(2840,2843,6),(2844,2844,6),(3160,3161,6),(3170,3171,6),(3172,3172,6),(3190,3199,6),(3229,3229,6),(3260,3260,6),(3262,3263,6),(3269,3269,6),(3230,3231,6),(3630,3639,6),(3750,3751,6),(3800,3800,6),(3860,3861,6),(3870,3873,6),(3910,3911,6),(3914,3914,6),(3915,3915,6),(3960,3962,6),(3991,3991,6),(3995,3995,6),
(2300,2390,7),(3020,3021,7),(3100,3111,7),(3130,3131,7),(3140,3149,7),(3150,3151,7),(3963,3965,7),
(2830,2830,8),(2831,2831,8),(2833,2833,8),(2834,2834,8),(2835,2835,8),(2836,2836,8),(3693,3693,8),(3840,3849,8),(3850,3851,8),(8000,8099,8),
(2800,2809,9),(2810,2819,9),(2820,2829,9),(2850,2859,9),(2860,2869,9),(2870,2879,9),(2890,2899,9),
(2200,2269,10),(2270,2279,10),(2280,2284,10),(2290,2295,10),(2297,2297,10),(2298,2298,10),(2299,2299,10),(2393,2395,10),(2397,2399,10),
(800,899,11),(1500,1511,11),(1520,1529,11),(1530,1539,11),(1540,1549,11),(1600,1699,11),(1700,1799,11),(2400,2439,11),(2450,2459,11),(2490,2499,11),(2660,2661,11),(2950,2952,11),(3200,3200,11),(3210,3211,11),(3240,3241,11),(3250,3259,11),(3261,3261,11),(3264,3264,11),(3270,3275,11),(3280,3281,11),(3290,3293,11),(3295,3299,11),(3420,3429,11),(3430,3433,11),(3440,3441,11),(3442,3442,11),(3446,3446,11),(3448,3448,11),(3449,3449,11),(3450,3451,11),(3452,3452,11),(3490,3499,11),(3996,3996,11),
(3300,3300,12),(3310,3317,12),(3320,3325,12),(3330,3339,12),(3340,3341,12),(3350,3357,12),(3360,3369,12),(3370,3379,12),(3390,3399,12),
(3400,3400,13),(3443,3443,13),(3444,3444,13),(3460,3469,13),(3470,3479,13),(3510,3519,13),(3520,3529,13),(3530,3530,13),(3531,3531,13),(3532,3532,13),(3533,3533,13),(3534,3534,13),(3535,3535,13),(3536,3536,13),(3538,3538,13),(3540,3549,13),(3550,3559,13),(3560,3569,13),(3580,3580,13),(3581,3581,13),(3582,3582,13),(3585,3585,13),(3586,3586,13),(3589,3589,13),(3590,3599,13),
(3600,3600,14),(3610,3613,14),(3620,3621,14),(3623,3629,14),(3640,3644,14),(3645,3645,14),(3646,3646,14),(3648,3649,14),(3660,3660,14),(3690,3690,14),(3691,3692,14),(3699,3699,14),
(2296,2296,15),(2396,2396,15),(3010,3011,15),(3537,3537,15),(3647,3647,15),(3694,3694,15),(3700,3700,15),(3710,3710,15),(3711,3711,15),(3713,3713,15),(3714,3714,15),(3715,3715,15),(3716,3716,15),(3792,3792,15),(3790,3791,15),(3799,3799,15),
(3720,3720,16),(3721,3721,16),(3723,3724,16),(3725,3725,16),(3728,3729,16),(3730,3731,16),(3740,3743,16),
(1000,1009,17),(1010,1019,17),(1020,1029,17),(1030,1039,17),(1040,1049,17),(1050,1059,17),(1060,1069,17),(1070,1079,17),(1080,1089,17),(1090,1099,17),(1100,1119,17),(1400,1499,17),
(1200,1299,18),
(1300,1300,19),(1310,1319,19),(1320,1329,19),(1330,1339,19),(1370,1379,19),(1380,1380,19),(1381,1381,19),(1382,1382,19),(1389,1389,19),(2900,2912,19),(2990,2999,19),
(4900,4900,20),(4910,4911,20),(4920,4922,20),(4923,4923,20),(4924,4925,20),(4930,4931,20),(4932,4932,20),(4939,4939,20),(4940,4942,20),
(4800,4800,21),(4810,4813,21),(4820,4822,21),(4830,4839,21),(4840,4841,21),(4880,4889,21),(4890,4890,21),(4891,4891,21),(4892,4892,21),(4899,4899,21),
(7020,7021,22),(7030,7033,22),(7200,7200,22),(7210,7212,22),(7214,7214,22),(7215,7216,22),(7217,7217,22),(7218,7218,22),(7219,7219,22),(7220,7221,22),(7230,7231,22),(7240,7241,22),(7250,7251,22),(7260,7269,22),(7270,7290,22),(7291,7291,22),(7292,7299,22),(7300,7300,22),(7310,7319,22),(7320,7329,22),(7330,7339,22),(7340,7342,22),(7349,7349,22),(7350,7351,22),(7352,7352,22),(7353,7353,22),(7359,7359,22),(7360,7369,22),(7370,7372,22),(7374,7374,22),(7375,7375,22),(7376,7376,22),(7377,7377,22),(7378,7378,22),(7379,7379,22),(7380,7380,22),(7381,7382,22),(7383,7383,22),(7384,7384,22),(7385,7385,22),(7389,7390,22),(7391,7391,22),(7392,7392,22),(7393,7393,22),(7394,7394,22),(7395,7395,22),(7396,7396,22),(7397,7397,22),(7399,7399,22),(7500,7500,22),(7510,7519,22),(7520,7529,22),(7530,7539,22),(7540,7549,22),(7600,7600,22),(7620,7620,22),(7622,7622,22),(7623,7623,22),(7629,7629,22),(7630,7631,22),(7640,7641,22),(7690,7699,22),(8100,8199,22),(8200,8299,22),(8300,8399,22),(8400,8499,22),(8600,8699,22),(8700,8700,22),(8710,8713,22),(8720,8721,22),(8730,8734,22),(8740,8748,22),(8800,8899,22),(8900,8910,22),(8911,8911,22),(8920,8999,22),
(3570,3579,23),(3622,3622,23),(3661,3661,23),(3662,3662,23),(3663,3663,23),(3664,3664,23),(3665,3665,23),(3666,3666,23),(3669,3669,23),(3670,3679,23),(3680,3680,23),(3681,3681,23),(3682,3682,23),(3683,3683,23),(3684,3684,23),(3685,3685,23),(3686,3686,23),(3687,3687,23),(3688,3688,23),(3689,3689,23),(3695,3695,23),(3810,3810,23),(3811,3811,23),(3812,3812,23),(3820,3820,23),(3821,3821,23),(3822,3822,23),(3823,3823,23),(3824,3824,23),(3825,3825,23),(3826,3826,23),(3827,3827,23),(3829,3829,23),(3830,3839,23),(7373,7373,23),
(2440,2449,24),(2520,2549,24),(2600,2639,24),(2640,2659,24),(2670,2699,24),(2760,2761,24),(3220,3221,24),(3410,3412,24),(3950,3955,24),
(4000,4013,25),(4040,4049,25),(4100,4100,25),(4110,4119,25),(4120,4121,25),(4130,4131,25),(4140,4142,25),(4150,4151,25),(4170,4173,25),(4190,4199,25),(4200,4200,25),(4210,4219,25),(4220,4229,25),(4230,4231,25),(4240,4249,25),(4400,4499,25),(4500,4599,25),(4600,4699,25),(4700,4700,25),(4710,4712,25),(4720,4729,25),(4730,4739,25),(4740,4749,25),(4780,4780,25),(4782,4782,25),(4783,4783,25),(4784,4784,25),(4785,4785,25),(4789,4789,25),
(5000,5000,26),(5010,5015,26),(5020,5023,26),(5030,5039,26),(5040,5042,26),(5043,5043,26),(5044,5044,26),(5045,5045,26),(5046,5046,26),(5047,5047,26),(5048,5048,26),(5049,5049,26),(5050,5059,26),(5060,5060,26),(5063,5063,26),(5064,5064,26),(5065,5065,26),(5070,5078,26),(5080,5080,26),(5081,5081,26),(5082,5082,26),(5083,5083,26),(5084,5084,26),(5085,5085,26),(5086,5087,26),(5088,5088,26),(5090,5090,26),(5091,5092,26),(5093,5093,26),(5094,5094,26),(5099,5099,26),(5100,5100,26),(5110,5113,26),(5120,5122,26),(5130,5139,26),(5140,5149,26),(5150,5159,26),(5160,5169,26),(5170,5172,26),(5180,5182,26),(5190,5199,26),
(5200,5200,27),(5210,5219,27),(5220,5229,27),(5230,5231,27),(5250,5251,27),(5260,5261,27),(5270,5271,27),(5300,5300,27),(5310,5311,27),(5320,5320,27),(5330,5331,27),(5334,5334,27),(5340,5349,27),(5390,5399,27),(5400,5400,27),(5410,5411,27),(5412,5412,27),(5420,5429,27),(5430,5439,27),(5440,5449,27),(5450,5459,27),(5460,5469,27),(5490,5499,27),(5500,5500,27),(5510,5529,27),(5530,5539,27),(5540,5549,27),(5550,5559,27),(5560,5569,27),(5570,5579,27),(5590,5599,27),(5600,5699,27),(5700,5700,27),(5710,5719,27),(5720,5722,27),(5730,5733,27),(5734,5734,27),(5735,5735,27),(5736,5736,27),(5750,5799,27),(5900,5900,27),(5910,5912,27),(5920,5929,27),(5930,5932,27),(5940,5940,27),(5941,5941,27),(5942,5942,27),(5943,5943,27),(5944,5944,27),(5945,5945,27),(5946,5946,27),(5947,5947,27),(5948,5948,27),(5949,5949,27),(5950,5959,27),(5960,5969,27),(5970,5979,27),(5980,5989,27),(5990,5990,27),(5992,5992,27),(5993,5993,27),(5994,5994,27),(5995,5995,27),(5999,5999,27),
(5800,5819,28),(5820,5829,28),(5890,5899,28),(7000,7000,28),(7010,7019,28),(7040,7049,28),(7213,7213,28),
(6000,6000,29),(6010,6019,29),(6020,6020,29),(6021,6021,29),(6022,6022,29),(6023,6024,29),(6025,6025,29),(6026,6026,29),(6027,6027,29),(6028,6029,29),(6030,6036,29),(6040,6059,29),(6060,6062,29),(6080,6082,29),(6090,6099,29),(6100,6100,29),(6110,6111,29),(6112,6113,29),(6120,6129,29),(6130,6139,29),(6140,6149,29),(6150,6159,29),(6160,6169,29),(6170,6179,29),(6190,6199,29),(6200,6299,29),(6300,6300,29),(6310,6319,29),(6320,6329,29),(6330,6331,29),(6350,6351,29),(6360,6361,29),(6370,6379,29),(6390,6399,29),(6400,6411,29),(6500,6500,29),(6510,6510,29),(6512,6512,29),(6513,6513,29),(6514,6514,29),(6515,6515,29),(6517,6519,29),(6520,6529,29),(6530,6531,29),(6532,6532,29),(6540,6541,29),(6550,6553,29),(6590,6599,29),(6610,6611,29),(6700,6700,29),(6710,6719,29),(6720,6722,29),(6723,6723,29),(6724,6724,29),(6725,6725,29),(6726,6726,29),(6730,6733,29),(6740,6779,29),(6790,6791,29),(6792,6792,29),(6793,6793,29),(6794,6794,29),(6795,6795,29),(6798,6798,29),(6799,6799,29),
(4950,4959,30),(4960,4961,30),(4970,4971,30),(4990,4991,30),
]

import numpy as np
_ff_lo = np.array([r[0] for r in _FF30_RANGES])
_ff_hi = np.array([r[1] for r in _FF30_RANGES])
_ff_cd = np.array([r[2] for r in _FF30_RANGES])

def sic_to_ff30(sic):
    if pd.isnull(sic): return np.nan
    s = int(sic)
    hit = np.where((_ff_lo <= s) & (s <= _ff_hi))[0]
    return int(_ff_cd[hit[0]]) if len(hit) else 30

sic = crsp_raw[['siccd']].dropna().copy()
sic['ff30'] = sic['siccd'].map(sic_to_ff30)
print('전체 unmapped(정의에 없어 30행도 아닌 경우는 없음, 30=Other 처리)')
print('siccd 결측 비율:', round(crsp_raw['siccd'].isna().mean()*100,1),'%')
print('ff30 분포:')
print(sic['ff30'].value_counts().sort_index())

In [ ]:
# 29

ind_cols = [f'fvol_{c}' for c in FVOL_COLS]

ia = comp_dist[comp_dist['datadate'] <= '2024-12-31'][['gvkey','datadate','rdq','calq'] + ind_cols].copy()
ia['ym'] = pd.to_datetime(ia['datadate']).dt.to_period('M')

sic_m = crsp_raw[['permno','date','siccd']].copy()
sic_m['permno'] = sic_m['permno'].astype('int64')
sic_m['ym'] = pd.to_datetime(sic_m['date']).dt.to_period('M')
sic_m = sic_m.drop_duplicates(['permno','ym'])[['permno','ym','siccd']]

ial = ia.merge(lk[['gvkey','permno','linkdt','linkenddt']], on='gvkey', how='inner')
ial = ial[(ial['datadate'] >= ial['linkdt']) & (ial['datadate'] <= ial['linkenddt'])]
ial['permno'] = ial['permno'].astype('int64')
ial = ial.merge(sic_m, on=['permno','ym'], how='left')

ial = ial.merge(univ_fq[['permno','ym']].drop_duplicates().assign(in_univ=1),
                on=['permno','ym'], how='left')
ial.loc[ial['in_univ'] != 1, ind_cols] = np.nan

ial['ff30'] = ial['siccd'].map(sic_to_ff30)

for c in ind_cols:
    ial[c + '_ia'] = ial[c] - ial.groupby(['calq','ff30'])[c].transform('mean')

ia_cols = [c + '_ia' for c in ind_cols]

ranked = ial.groupby('calq')[ia_cols].rank(pct=True)
ranked.columns = [f'rank_{x}' for x in FVOL_COLS]
ial = pd.concat([ial.reset_index(drop=True), ranked.reset_index(drop=True)], axis=1)
rcols = [f'rank_{x}' for x in FVOL_COLS]
ial['n_valid_rank'] = ial[rcols].notna().sum(axis=1)
ial['FVOL_ia'] = ial[rcols].mean(axis=1)
ial.loc[ial['n_valid_rank'] < 10, 'FVOL_ia'] = np.nan

fvol_ia = ial[ial['FVOL_ia'].notna()][['gvkey','datadate','rdq','FVOL_ia']].copy()
print('FVOL_ia valid firm-quarters:', len(fvol_ia))
print('avg per calq:', round(ial[ial['FVOL_ia'].notna()].groupby('calq').size().mean(), 0))

fv_ia = fvol_ia.merge(link, on='gvkey', how='inner')
fv_ia = fv_ia[(fv_ia['datadate'] >= fv_ia['linkdt']) & (fv_ia['datadate'] <= fv_ia['linkenddt'])]
fv_ia = fv_ia.dropna(subset=['rdq']).copy()
fv_ia['permno'] = fv_ia['permno'].astype('int64')
fv_ia['avail'] = pd.to_datetime(fv_ia['rdq']).dt.to_period('Q').dt.end_time.dt.normalize()
fv_ia = fv_ia.sort_values('avail')

qends = crsp[crsp['date'].dt.month.isin([3,6,9,12]) & crsp['ret'].notna() & (crsp['date'] >= '1976-10-01')][['permno','date','prc']].copy()
qends = qends.rename(columns={'date':'form_date'}).sort_values('form_date')

formed_ia = pd.merge_asof(
    qends, fv_ia[['permno','avail','FVOL_ia']].sort_values('avail'),
    left_on='form_date', right_on='avail', by='permno', direction='backward')
formed_ia = formed_ia.dropna(subset=['FVOL_ia'])
formed_ia = formed_ia[formed_ia['prc'].abs() >= 5].copy()
formed_ia['decile'] = formed_ia.groupby('form_date')['FVOL_ia'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1)
formed_ia = formed_ia.dropna(subset=['decile'])
formed_ia['decile'] = formed_ia['decile'].astype(int)
print('formation obs:', len(formed_ia))
print('avg firms/formation:', round(formed_ia.groupby('form_date').size().mean(), 0))

In [ ]:
# 30

fm_ia = formed_ia[['permno','form_date','decile']].copy()
fm_ia['form_ym'] = pd.to_datetime(fm_ia['form_date']).dt.to_period('M')

holds_ia = []
for k in [1, 2, 3]:
    h = fm_ia.copy()
    h['hold_ym'] = h['form_ym'] + k
    holds_ia.append(h)
hold_ia = pd.concat(holds_ia, ignore_index=True)

rp_ia = crsp[['permno','date','ret_adj','mktcap']].copy().sort_values(['permno','date'])
rp_ia['w'] = rp_ia.groupby('permno')['mktcap'].shift(1)
rp_ia['ym'] = pd.to_datetime(rp_ia['date']).dt.to_period('M')
mm_ia = rp_ia.merge(hold_ia, left_on=['permno','ym'], right_on=['permno','hold_ym'], how='inner')
mm_ia = mm_ia.dropna(subset=['ret_adj','w']).drop_duplicates(['permno','form_date','ym'])

dec_ia = (mm_ia.groupby(['ym','decile'])
          .apply(lambda g: (g['ret_adj'] * g['w']).sum() / g['w'].sum(), include_groups=False)
          .reset_index(name='vw'))
wide_ia = dec_ia.pivot(index='ym', columns='decile', values='vw')
lh_ia = (wide_ia[1] - wide_ia[10]).dropna()

print('IA factor Low-High: {:.2f}% t={:.2f}'.format(lh_ia.mean()*100, lh_ia.mean()/(lh_ia.std()/np.sqrt(len(lh_ia)))))

ffm_ia = ff5.copy()
ffm_ia['ym'] = pd.to_datetime(ffm_ia['date']).dt.to_period('M')
ffm_ia = ffm_ia[['ym','mktrf','smb','hml','rmw','cma']]
qmf_ia = qf.rename(columns={'R_MKT':'qmkt','R_ME':'qme','R_IA':'qia','R_ROE':'qroe'})[['ym','qmkt','qme','qia','qroe']]

dsp_ia = lh_ia.rename('fvol').reset_index().merge(ffm_ia, on='ym', how='left').merge(qmf_ia, on='ym', how='left')
for c in dsp_ia.columns:
    if c != 'ym':
        dsp_ia[c] = pd.to_numeric(dsp_ia[c], errors='coerce')

def sharpe2_ia(R):
    R = np.asarray(R, float)
    mu = R.mean(axis=0)
    cov = np.cov(R, rowvar=False)
    return mu @ np.linalg.solve(cov, mu)

def spanning_ia(base_cols, label):
    sub = dsp_ia.dropna(subset=['fvol'] + base_cols)
    y = sub['fvol'].to_numpy(dtype=float)
    X = sm.add_constant(sub[base_cols].to_numpy(dtype=float))
    m = sm.OLS(y, X).fit()
    a, ta = m.params[0]*100, m.tvalues[0]
    rbar = sub['fvol'].mean()*100
    tr = sub['fvol'].mean()/(sub['fvol'].std()/np.sqrt(len(sub)))
    sr0 = sharpe2_ia(sub[base_cols].to_numpy(dtype=float))
    sr1 = sharpe2_ia(sub[['fvol']+base_cols].to_numpy(dtype=float))
    print(f'{label}: rbar {rbar:.2f}%(t{tr:.2f})  alpha {a:.2f}%(t{ta:.2f})  SR2_0 {sr0:.3f} SR2 {sr1:.3f}  IR2 {sr1-sr0:.3f}')

print('=== Panel A: vs FF5 ===  (paper: alpha 0.89% t6.29, SR2 0.11->0.18)')
spanning_ia(['mktrf','smb','hml','rmw','cma'], 'FVOL_ia')
print('=== Panel B: vs q4 ===  (paper: alpha 0.86% t5.82, SR2 0.14->0.21)')
spanning_ia(['qmkt','qme','qia','qroe'], 'FVOL_ia')